# SC26 DragonHPC Multiprocessing, Native, and DDict Tutorial

**Estimated time:** ~30-45 minutes  
**Format:** 6 short exercises - each has a problem and a hidden answer you can reveal.

---

## What you will learn

This notebook introduces Python multiprocessing with DragonHPC for single-node
and multi-node runs. You will practice how to:

- launch processes
- synchronize worker execution
- communicate data between processes
- observe process placement across nodes

The exercises are beginner friendly, while also connecting to concepts that
intermediate and advanced users already know from Python multiprocessing.

---

## Setup - run this first!


In [1]:
import os
import socket
import time

import dragon
import multiprocessing as mp

# Shim for this tutorial environment; eliminated in future Dragon release
import dragon.workflows.runtime as runtime
dragon._patch_multiprocessing()
dragon_rt = runtime.start_local_runtime(timeout=5)
runtime.attach(runtime=dragon_rt, auth_token=dragon_rt.auth_token)
dragon.native.pool.current_process = mp.current_process

mp.set_start_method("dragon")

print("Dragon multiprocessing ready.")

info={'node_index': 0, 'gs_qd': 'BV8AAAAAAAAAAgAAAAAAAAA3AAAAAAAAAAAAAAAAAAAAcfNluE+0TwcFLtd6GM2leAAAAAAAAAAA4NjABAAAAAAvREQwMDAxMzVpbmYwTQADAIAAAAAAAAAAAAAAAAAACOABAAAAAAA=', 'ls_qd': 'BV8AAAAAAAAAAAAAAAAAAEA3AAAAAAAAAAAAAAAAAAAAcfNluE+0TwcFLtd6GM2leAAAAAAAAAAA4NjABAAAAAAvREQwMDAxMzVpbmYwTQAAACAAAAAAAAAAAAAAAAAACOABAAAAAAA=', 'default_pd': 'AAAAAAAAAEBx82W4T7RPBwUu13oYzaV4AAAAAAAAAACA6QAHAAAAAC9ERDAwMDEzNWRlZjBNAA==', 'inf_pd': 'AAAAAAAAAABx82W4T7RPBwUu13oYzaV4AAAAAAAAAADg2MAEAAAAAC9ERDAwMDEzNWluZjBNAA==', 'rt_uid': '8693580160732442117', 'auth_token': 'c047cb39c9de6919a17499113a4cfcace0aead0f9c1bd8036411a6a1d95be510'}
Dragon multiprocessing ready.


`multiprocessing` synchronization and communication objects (for example
`Queue`, `Event`, `Lock`, `Value`) happily work with `"dragon"` set as
the start method in `multiprocessing` just as they would with any of the
other start methods in `multiprocessing` (i.e. `"spawn"`, `"fork"`, or
`"forkserver"`).

When the Dragon runtime is available, a number of opportunities for
optimization arise which is why there are additional tools available
in the `dragon` module.  After exercising the use of Dragon through
standard `multiprocessing`, we will begin by introducing the Dragon
Native `Process` object which exposes further control over the creation
and placement of processes, noting how it intentionally mirrors the
familiar multiprocessing API patterns.

---

## Exercise 1 - Parallel map with multiprocessing Pool

**Background:**

`multiprocessing.Pool` is useful when the same function should run over many
inputs. With Dragon configured as the multiprocessing backend, you can use
`Pool.map` for data-parallel workloads executing across many nodes and all
communication will travel over Dragon's HSTA (High Speed Transfer Agent).

Ref: https://docs.python.org/3/library/multiprocessing.html#introduction

Demo pattern:

```python
def double(x):
    return 2 * x

with mp.Pool(processes=4) as pool:
    out = pool.map(double, [1, 2, 3, 4])

print(out)  # [2, 4, 6, 8]
```

**Your task:**

1. Write a function `transform(x)` that returns `x * x + 1`
2. Use `mp.Pool(processes=4)`
3. Call `pool.map(transform, data)` where `data = list(range(12))`
4. Print the mapped results
5. Print the sum of the mapped results

In [2]:
# -- Exercise 1 -- your code here -----------------------------------------------
def transform(x):
    return x * x + 1

data = list(range(12))

with mp.Pool(processes=4) as pool:
    mapped = pool.map(transform, data)

print(mapped)
print("sum:", sum(mapped))

[1, 2, 5, 10, 17, 26, 37, 50, 65, 82, 101, 122]
sum: 518


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def transform(x):
    return x * x + 1

data = list(range(12))

with mp.Pool(processes=4) as pool:
    mapped = pool.map(transform, data)

print(mapped)
print("sum:", sum(mapped))
```

Expected mapped output:
```
[1, 2, 5, 10, 17, 26, 37, 50, 65, 82, 101, 122]
sum: 518
```

</details>

---

## Exercise 2 - Launch processes and join, using queues to communicate

**Background:**

A `mp.Process` is started with `start()` and waited on with `join()`. This is the
first basic building block for parallel programs though in contrast to the
prior example employing `mp.Pool`, the return values from target
functions are not collected and returned to the caller.  This motivates the
use of `mp.Queue` for process-safe message passing which, among many other uses,
feels like an especially natural fit for producer-consumer code patterns.

By setting `multiprocessing`'s start method to `"dragon"`, all of these tools
from the `multiprocessing` module will leverage the Dragon runtime.  But Dragon
also offers tools to unlock opportunities for optimization, tuning, and
control that `multiprocessing` does not.  The first basic example of this is
the Dragon Native process object, `dragon.native.Process`, which supports the
standard `mp.Process` API but expands on it as we will see in a later exercise.

Here is the core Dragon Native process pattern:

```python
def hello_worker(name):
    print(f"hello from {name}, pid={os.getpid()}")  # stdout is not generally a best practice for collection

p = dragon.native.Process(target=hello_worker, args=("dragon",))
p.start()
p.join()
```

Here is the core pattern for `Queue`:

```python
q = mp.Queue()

def say_hello(q):
    q.put(("msg", "hello from worker"))

p = dragon.native.Process(target=say_hello, args=(q,))  #  mp.Process works here too
p.start()
message = q.get()
p.join()
print(message)
```


**Your task:**

1. Write a worker function `square_worker(i, q)` that accepts an `int` i and `Queue` q, constructs a nicely formmated `str` containing the worker id `i`, PID, and `i*i`, and places that string into the queue.
2. Launch 4 processes using `dragon.native.Process` (though `mp.Process` would also work; try both?)
3. Start all processes, read and print the 4 messages placed into the queue by the workers, then join all processes
4. Print `All workers finished` at the end

In [3]:
# -- Exercise 2 -- your code here -----------------------------------------------
def square_worker(i, q):
    q.put(f"worker={i} pid={os.getpid()} square={i*i}")

q = mp.Queue()
procs = [dragon.native.Process(target=square_worker, args=(i, q)) for i in range(4)]

for p in procs:
    p.start()

for p in procs:
    print(q.get())

for p in procs:
    p.join()

print("All workers finished")

worker=0 pid=52321 square=0
worker=2 pid=52323 square=4
worker=1 pid=52322 square=1
worker=3 pid=52324 square=9
All workers finished


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def square_worker(i, q):
    q.put(f"worker={i} pid={os.getpid()} square={i*i}")

q = mp.Queue()
procs = [dragon.native.Process(target=square_worker, args=(i, q)) for i in range(4)]

for p in procs:
    p.start()

for p in procs:
    print(q.get())

for p in procs:
    p.join()

print("All workers finished")
```

</details>

---

## Exercise 3 - Phase synchronization with Barrier

**Background:**

A `Barrier` blocks each worker until a fixed number of participants arrive.
This is useful for staged workflows where all workers must reach the same
checkpoint before continuing together.

With Dragon, `Barrier` objects work in single-node or multi-node
execution environments.

Demo pattern:

```python
barrier = mp.Barrier(2)

def waiter(barrier):
    print("before barrier")
    barrier.wait()
    print("after barrier")

p = dragon.native.Process(target=waiter, args=(barrier,))
p.start()
time.sleep(0.5)
barrier.wait()
p.join()
```

**Your task:**

1. Create a `Barrier` named `phase_barrier` with 6 participants (5 workers + parent)
2. Write worker code that prints `ready`, waits on the barrier, then prints `running`
3. Launch 5 workers
4. Sleep for 2 seconds in the parent and then call `phase_barrier.wait()`
5. Join workers

In [4]:
# -- Exercise 3 -- your code here -----------------------------------------------
def staged_worker(worker_id, phase_barrier):
    print(f"worker={worker_id} ready")
    phase_barrier.wait()
    print(f"worker={worker_id} running")

phase_barrier = mp.Barrier(6)
procs = [dragon.native.Process(target=staged_worker, args=(i, phase_barrier)) for i in range(5)]

for p in procs:
    p.start()

time.sleep(2.0)
print("Parent arriving at barrier")
phase_barrier.wait()

for p in procs:
    p.join()

Parent arriving at barrier


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def staged_worker(worker_id, phase_barrier):
    print(f"worker={worker_id} ready")
    phase_barrier.wait()
    print(f"worker={worker_id} running")

phase_barrier = mp.Barrier(6)
procs = [dragon.native.Process(target=staged_worker, args=(i, phase_barrier)) for i in range(5)]

for p in procs:
    p.start()

time.sleep(2.0)
print("Parent arriving at barrier")
phase_barrier.wait()

for p in procs:
    p.join()
```

</details>

---

## Exercise 4 - Control CPU affinity with Dragon Policy

**Background:**

Dragon `Policy` lets you control affinity. For CPUs, use `cpu_affinity`.
For example, `Policy(cpu_affinity=[1])` requests placement targeting CPU 1.
A `Policy` can be used with `dragon.native.Process` but not with `mp.Process`.

Demo pattern:

```python
def report_affinity(out_q):
    out_q.put((os.getpid(), os.environ.get("CUDA_VISIBLE_DEVICES", "unset")))

q = mp.Queue()
pol = Policy(cpu_affinity=[0])
p = dragon.native.Process(target=report_affinity, args=(q,), policy=pol)
p.start()
print(q.get())
p.join()
```

**Your task:**

1. Create two policies: one for CPU 0 and one for CPU 1
2. Launch two workers with those explicit policies
3. Each worker should report `(rank, pid, CUDA_VISIBLE_DEVICES)`
4. Print both reports

If your system has fewer than 2 CPUs, use CPU fallback and still run both
workers (the environment report may be the same).

In [5]:
# -- Exercise 4 -- your code here -----------------------------------------------
def affinity_worker(rank, out_q):
    out_q.put((rank, os.getpid(), os.environ.get("CUDA_VISIBLE_DEVICES", "unset")))

q = mp.Queue()
num_cpus = mp.cpu_count()
cpu0 = 0
cpu1 = 1 if num_cpus > 1 else 0

policies = [dragon.Policy(cpu_affinity=[cpu0]), dragon.Policy(cpu_affinity=[cpu1])]
procs = [
    dragon.native.Process(target=affinity_worker, args=(0, q), policy=policies[0]),
    dragon.native.Process(target=affinity_worker, args=(1, q), policy=policies[1]),
]

for p in procs:
    p.start()

for _p in procs:
    print(q.get())

for p in procs:
    p.join()

(1, 52538, 'unset')
(0, 52537, 'unset')


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def affinity_worker(rank, out_q):
    out_q.put((rank, os.getpid(), os.environ.get("CUDA_VISIBLE_DEVICES", "unset")))

q = mp.Queue()
num_cpus = mp.cpu_count()
cpu0 = 0
cpu1 = 1 if num_cpus > 1 else 0

policies = [dragon.Policy(cpu_affinity=[cpu0]), dragon.Policy(cpu_affinity=[cpu1])]
procs = [
    dragon.native.Process(target=affinity_worker, args=(0, q), policy=policies[0]),
    dragon.native.Process(target=affinity_worker, args=(1, q), policy=policies[1]),
]

for p in procs:
    p.start()

for _p in procs:
    print(q.get())

for p in procs:
    p.join()
```

</details>

---

## Exercise 5 — Create, write, read, and destroy a `DDict`

**Background:**

A `DDict` is Dragon's **distributed in-memory key-value store**. It looks and
feels like a Python `dict`, but the data lives across **one or more manager
processes** that can span multiple nodes. The best part is that the organization
of the data is transparent to the user. When user code interacts with it, the
code does not need to know the location of any key-value pair stored in it.

<img src="../images/ddict_overview.png" width="80%">

Key characteristics:
- Standard `dict`-like interface (`d[key] = val`, `d[key]`, `del d[key]`, `len(d)`, ...)
- Values are transparently serialized with `cloudpickle` or a pickler of your choice
- Data is shared safely across **multiple processes** without locks
- Supports **checkpointing**, **persistent keys**, and **filtering** 
- Other functionality not covered here like **broadcast puts** and **batch puts**
- Data is automatically balanced across all the managers

Instantiating and using an empty `DDict` object is as easy as:

```python
from dragon import DDict

d = DDict()

d["greeting"] = "howdy"
d["meaning"] = 42
print(len(d))  # 2
```

**Your Task:**

1. Create a `DDict`
2. Store `{"cat": 1, "dog": 2, "bird": 3}`
3. Print all keys by iterating over `d.keys()`
4. Check whether `"dog"` and `"fish"` are in the dictionary
5. Delete `"cat"` and confirm the length dropped to 2
6. Use `d.get("fish", "not found")` and print the result
7. Destroy the dictionary (to release its allocated memory)

In [6]:
# ── Exercise 5 ── your code here ─────────────────────────────────────────────
d = dragon.DDict()

d["cat"]  = 1
d["dog"]  = 2
d["bird"] = 3

print("keys:", list(d.keys()))          # ['cat', 'dog', 'bird'] (order may vary)
print("dog in d:", "dog" in d)          # True
print("fish in d:", "fish" in d)        # False

del d["cat"]
print("len after delete:", len(d))      # 2

print(d.get("fish", "not found"))       # not found

d.destroy()

keys: ['bird', 'dog', 'cat']
dog in d: True
fish in d: False
len after delete: 2
not found


<details>
<summary><b>▶ Show Answer</b></summary>

```python
d = dragon.DDict()

d["cat"]  = 1
d["dog"]  = 2
d["bird"] = 3

print("keys:", list(d.keys()))          # ['cat', 'dog', 'bird'] (order may vary)
print("dog in d:", "dog" in d)          # True
print("fish in d:", "fish" in d)        # False

del d["cat"]
print("len after delete:", len(d))      # 2

print(d.get("fish", "not found"))       # not found

d.destroy()
```

</details>

---

## Exercise 6 — Sharing a `DDict` across processes

**Background:**

A `DDict` is serializable (`pickle`-able). Passing a `DDict` to a distinct process does not cause an expensive transfer or serialization of any its contents (the keys or values).  Instead, that process will share the **same** `DDict`, capable of being accessed or mutated by both child and parent.

When instantiating a `DDict` object, you may specify the number of managers and nodes
on which the `DDict` will hold its distributed data. A `DDict` automatically starts
up a specified number of managers and an orchestrator (not shown in the picture) and
user client code interacts with these processes to transparently provide you with a
distributed key-value store that can scale to any size and automatically distributes
your data evenly across all the mangers.


```python
import dragon.native

def worker(d, key, value):
    d[key] = value   # writes into the shared DDict
    d["hello"] = "world" # keys and values don't have to be passed to child proceses

d = DDict(1, 1, 32 * 1024 * 1024) # DDict(managers_per_node, n_nodes, total_mem_bytes)
proc = dragon.native.Process(target=worker, args=(d, "x", 99)) # or use mp.Process
proc.start() # starts the process
proc.join()  # waits for the process to exit
print(d["hello"]) # and then prints world
d.destroy()
```

This tells Dragon to instantiate and bring up a `DDict` with `managers_per_node * n_nodes` managers. The `total_mem_bytes` is spread across all the managers. For more information on creating `DDict` objects [see the DDict API](https://dragonhpc.github.io/dragon/doc/_build/html/ref/data/dragon.data.DDict.html#dragon.data.DDict). 

- `managers_per_node=1`, `n_nodes=1` → one manager, single node
- `total_mem_bytes=64*1024*1024` → 64 MB pool
- Remember to call `d.destroy()` to release resources when finished


**Your Task:**

Write a function `count_words(label, text, d)` that splits `text` on spaces and stores
`d[label] = count` as a count of the words in the supplied text.

Hint: Calling `s.split()` on a `str` object `s` in Python will return a list of all the space 
separated *words* of the string.

Then:
1. Create a `DDict` (2 managers, 1 node, 64 MB)
2. Launch **3 processes**, each calling `count_words` with a different song lyric
3. After all processes finish, print every key-value pair in the dictionary
4. Destroy the dictionary

Song lyrics and associated labels to use:
```
("never gonna give you up never gonna let you down", "rick")
("hello is it me you're looking for", "lionel")
("cause I'll be rocking this party eight days a week", "beastie")
```

In [7]:
# ── Exercise 6 ── your code here ─────────────────────────────────────────────
lyrics_and_labels = [
    ("never gonna give you up never gonna let you down", "rick"),
    ("hello is it me you're looking for", "lionel"),
    ("cause I'll be rocking this party eight days a week", "beastie"),
]

def count_words(label, text, d):
    d[label] = len(text.split())

d = dragon.DDict(2, 1, 64 * 1024 * 1024)

procs = [dragon.native.Process(target=count_words, args=(l, t, d)) for t, l in lyrics_and_labels]
for p in procs:
    p.start()
for p in procs:
    p.join()

for k, v in d.items():
    print(f"  {k}: {v}")

d.destroy()

  rick: 10
  beastie: 10
  lionel: 7


<details>
<summary><b>▶ Show Answer</b></summary>

```python
lyrics_and_labels = [
    ("never gonna give you up never gonna let you down", "rick"),
    ("hello is it me you're looking for", "lionel"),
    ("cause I'll be rocking this party eight days a week", "beastie"),
]

def count_words(label, text, d):
    d[label] = len(text.split())

d = dragon.DDict(2, 1, 64 * 1024 * 1024)

procs = [dragon.native.Process(target=count_words, args=(l, t, d)) for t, l in lyrics_and_labels]
for p in procs:
    p.start()
for p in procs:
    p.join()

for k, v in d.items():
    print(f"  {k}: {v}")

d.destroy()
```

<br />

**Synchronizing Updates to the Same Key in Parallel** 

Because multiple processes may attempt to store an updated value for a key which depends upon the
existing associated value, care must be taken because whichever process writes last wins.  Depending
upon your code's needs, look at some of the functionality available on the `DDict` that has no
equivalent on the standard `dict` such as the atomic update `fetch_add` method to add values to
the existing stored value.

For example, the `fetch_add` will operate atomically to add an integer value to a specified key/value pair. 

```python
def count_words(d, text):
    for word, count in Counter(text.split()).items():
        d.fetch_add(word, count)
```

Using `fetch_add` to retrieve fetch/added values works in both Python and C++ Dragon APIs.
</details>

---

## Summary

You used DragonHPC with familiar multiprocessing concepts to launch workers,
synchronize phases, communicate data, and run data-parallel map operations.
The same API patterns apply whether you run on one node or many.

| Concept | API |
|---|---|
| Start Dragon backend | `mp.set_start_method("dragon")` |
| Launch process | `mp.Process` and/or `dragon.native.Process(target=..., args=...)` |
| Wait for completion | `p.join()` |
| Mutual exclusion | `mp.Lock()` |
| Shared scalar | `mp.Value(...)` |
| Message passing | `mp.Queue()` |
| Phase synchronization | `mp.Barrier(n)` |
| Pool map | `mp.Pool(...).map(func, data)` |
| Multi-node placement check | `socket.gethostname()` |

### What next?

Try adapting these exercises to your own application by replacing toy data with
real workloads and scaling the worker count to your full Dragon allocation.